# Statistical Tests

- Sampling approach: Monte Carlo stratified sampling with partitioned datasets.

## Hypothesis

- Statement: biber dimensions can be derived from an ensemble of local, small zero-shot models to give the same biber dimensions derived from traditional rule-based features. 
- The null hypothesis (H0) is that MDA Dimensions == Zero-Shot Dimensions. The alternative hypothesis is that MDA Dimensions != Zero-Shot Dimensions.
- If we fail to reject the null hypothesis then we are proving that they are the same. 

## Tests


| Test                             | Purpose                         | Data Scale           | Paired?                  | What It Tells You                         | Pros                                        | Cons                                               | Decision Rule (Possible Difference) |
| -------------------------------- | ------------------------------- | -------------------- | ------------------------ | ----------------------------------------- | ------------------------------------------- | -------------------------------------------------- | ------------- |
| **Paired t-test**                | Test mean difference            | Continuous           | Yes                      | Whether average difference = 0            | Simple, powerful, widely understood         | Assumes normality; cannot prove equivalence | p < 0.05 -> reject H0 |
| **Wilcoxon signed-rank**         | Test median difference          | Ordinal / continuous | Yes                      | Whether median difference = 0             | Non-parametric, robust to outliers          | Less power than t-test; still not equivalence | p < 0.05 -> reject H0 |
| **Pearson correlation (r)**      | Linear association              | Continuous           | No(but paired inputs)    | Whether scores vary together linearly     | Easy to interpret; widely used              | High r ≠ agreement; ignores bias | Low r -> potential disagreement |
| **Spearman correlation (ρ)**     | Rank association                | Ordinal / continuous | No                       | Whether rankings agree                    | Robust to non-normality                     | Ignores scale differences | Low p -> potential difference |

- Paired t-test and Wilcoxon directly test the null hypothesis.
- Pearson / Spearman check correlation patterns between dimensions across seeds/factors.

## Methodology
- Compute and aggregate train, test, val. 
- Compute and aggregate on seed.
- Compute and aggregate on dataset.
- Compute and aggregate on factor. 

In [1]:
import os
import torch
import random
import numpy as np

import pandas as pd
import polars as pl

from scipy.stats import ttest_rel, wilcoxon, pearsonr, spearmanr, combine_pvalues

In [2]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [3]:
BASE_DIR = 'outputsAll'
FACTOR_COLS = [f'factor_{i}' for i in range(1, 7)]

In [4]:
# Load all data.
all_data = []
for seed in os.listdir(BASE_DIR):
    seed_path = os.path.join(BASE_DIR, seed)
    if not os.path.isdir(seed_path):
        continue
        
    mda_file = os.path.join(seed_path, 'mda_dim_scores.csv')
    zs_file = os.path.join(seed_path, 'mean_model_zero_shot_classification.csv')
    if not os.path.exists(mda_file) or not os.path.exists(zs_file):
        continue
        
    mda_df = pd.read_csv(mda_file)
    zs_df = pd.read_csv(zs_file)
        
    # Assign doc_cat from MDA to Zero-Shot.
    zs_df['doc_cat'] = mda_df['doc_cat']

    # Extract factor columns.
    mda_factors = mda_df[FACTOR_COLS]
    zs_factors = zs_df[FACTOR_COLS]
        
    # Loop over datasets
    for dataset in mda_df['doc_cat'].unique():
        mda_subset = mda_factors[mda_df['doc_cat'] == dataset]
        zs_subset = zs_factors[zs_df['doc_cat'] == dataset]
            
        for factor in FACTOR_COLS:
            all_data.append({
                'seed': seed,
                'dataset': dataset,
                'factor': factor,
                'mda': mda_subset[factor].mean(),
                'zeroshot': zs_subset[factor].mean()
            })

df = pd.DataFrame(all_data)
df['diff'] = df['mda'] - df['zeroshot']

# Statistical analysis.
results = []

for dataset in df['dataset'].unique():
    for factor in df['factor'].unique():
        subset = df[(df['dataset'] == dataset) & (df['factor'] == factor)]
        mda_vals = subset['mda'].values
        zs_vals = subset['zeroshot'].values
        diff_vals = subset['diff'].values
        
        # Paired t-test.
        try:
            t_stat, t_p = ttest_rel(mda_vals, zs_vals)
        except Exception as e:
            print(f"Error: {e}")
            t_p = np.nan
        
        # Wilcoxon signed-rank.
        try:
            w_stat, w_p = wilcoxon(diff_vals)
        except Exception as e:
            print(f"Error: {e}")
            w_p = np.nan
        
        # Pearson correlation.
        try:
            pearson_r, pearson_p = pearsonr(mda_vals, zs_vals)
        except Exception as e:
            print(f"Error: {e}")
            pearson_r, pearson_p = np.nan, np.nan
        
        # Spearman correlation.
        try:
            spearman_rho, spearman_p = spearmanr(mda_vals, zs_vals)
        except Exception as e:
            print(f"Error: {e}")
            spearman_rho, spearman_p = np.nan, np.nan
        
        # Store results.
        results.append({
            'dataset': dataset,
            'factor': factor,
            'mean_diff': diff_vals.mean(),
            'paired_t_p': t_p,
            'wilcoxon_p': w_p,
            'pearson_r': pearson_r,
            'pearson_p': pearson_p,
            'spearman_rho': spearman_rho,
            'spearman_p': spearman_p
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['dataset', 'factor']).reset_index(drop=True)
print(pl.from_pandas(results_df))

final_results = []

for factor in FACTOR_COLS:
    factor_subset = results_df[results_df['factor'] == factor]
    
    final_results.append({
        'factor': factor,
        'mean_diff': factor_subset['mean_diff'].mean(),
        'paired_t_p_mean': factor_subset['paired_t_p'].mean(),  # Average p-value for reference
        'wilcoxon_p_mean': factor_subset['wilcoxon_p'].mean(),
        'pearson_r_mean': factor_subset['pearson_r'].mean(),
        'spearman_rho_mean': factor_subset['spearman_rho'].mean()
    })

final_df = pd.DataFrame(final_results)
final_df = final_df.sort_values('factor').reset_index(drop=True)

print(pl.DataFrame(final_results))

shape: (60, 9)
┌─────────┬──────────┬───────────┬────────────┬───┬───────────┬───────────┬────────────┬───────────┐
│ dataset ┆ factor   ┆ mean_diff ┆ paired_t_p ┆ … ┆ pearson_r ┆ pearson_p ┆ spearman_r ┆ spearman_ │
│ ---     ┆ ---      ┆ ---       ┆ ---        ┆   ┆ ---       ┆ ---       ┆ ho         ┆ p         │
│ str     ┆ str      ┆ f64       ┆ f64        ┆   ┆ f64       ┆ f64       ┆ ---        ┆ ---       │
│         ┆          ┆           ┆            ┆   ┆           ┆           ┆ f64        ┆ f64       │
╞═════════╪══════════╪═══════════╪════════════╪═══╪═══════════╪═══════════╪════════════╪═══════════╡
│ atis    ┆ factor_1 ┆ -0.701015 ┆ 4.4961e-51 ┆ … ┆ 0.290449  ┆ 0.003374  ┆ 0.2809     ┆ 0.004641  │
│ atis    ┆ factor_2 ┆ 0.044906  ┆ 0.025428   ┆ … ┆ 0.056931  ┆ 0.573702  ┆ 0.05991    ┆ 0.553784  │
│ atis    ┆ factor_3 ┆ -0.15909  ┆ 8.2990e-7  ┆ … ┆ 0.11918   ┆ 0.237592  ┆ 0.181086   ┆ 0.071383  │
│ atis    ┆ factor_4 ┆ 0.44123   ┆ 2.0691e-25 ┆ … ┆ -0.117482 ┆ 0.244389  ┆ 

# Reject the null hypothesis; this means that Biber Dimensions and Zero-Shot Dimensions are statistically significantly different.